# 4 — FileList for the processed profile tables

Builds `processed_profiles-FileList.tsv`, the BioStudies file list for the
`processed_profiles/` folder of **S-BIAD2254** — the tier the figures actually read.
The images and segmentation masks are described by `spher_colo52-FileList.tsv`, written
by `4_ImageBioArchive_Metadata.ipynb`; this is the same job for the derived tables.

The authority for *which* files belong in the deposit is
`scripts/data_manifest.tsv`, which `download_data.py --write-manifest data` hashes
straight out of `data/`. Deriving the list from it rather than from a directory walk
means the file list, the checksums the downloader verifies against, and the upload can
never disagree.


In [ ]:
import sys, pathlib, csv, re
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import DATA_ROOT

import pandas as pd

MANIFEST = ROOT / "scripts" / "data_manifest.tsv"
DEST_FOLDER = "processed_profiles"          # folder name inside the deposit
OUT = pathlib.Path("processed_profiles-FileList.tsv")

rows = [r for r in csv.DictReader(
    [l for l in MANIFEST.open() if not l.startswith("#")], delimiter="\t")]
print(f"{len(rows)} files in the manifest")

## Attributes

BioStudies shows these as facets, so a reader can filter the deposit by experiment or by
which panel a table feeds. The image FileList carries fourteen such columns; a bare list
of paths would be valid but much less useful.


In [ ]:
EXPERIMENTS = {
    "exp1_main":          "3D spheroid Cell Painting, 52 compounds, z-slice sampling",
    "exp2_spheroid_size": "Seeding density / spheroid size",
    "exp3_clearing_mag_z": "Clearing, magnification, z-sampling density",
    "exp4_objective":     "Air vs water-immersion objective",
}


def describe(path):
    """(data_type, condition, used_by) for one manifest path."""
    name = pathlib.Path(path).name
    exp = path.split("/")[0]

    if path.startswith("exp1_main/slices/"):
        return ("Feature-selected profiles, z-subsampled",
                name[len("selected_"):-len(".parquet")], "Suppl Fig 3a")
    if name.startswith("normalized_data_"):
        return ("Normalised per-slice profiles",
                "HCT116; per-plate and per-plate-and-slice", "Fig 2g")
    if m := re.match(r"grit_data_(\w+)_(HCT116|HT29)\.parquet$", name):
        return ("Grit scores", f"{m.group(2)}, {m.group(1)}",
                "Fig 3c-h, Fig 4, Fig 5, Fig 6e")
    if m := re.match(r"selected_data_(\w+)_(HCT116|HT29)\.parquet$", name):
        return ("Feature-selected profiles", f"{m.group(2)}, {m.group(1)}",
                "grit-score input; Suppl Fig 3, Suppl Fig 5")
    if name == "selected_data_HT29.parquet":
        return ("Feature-selected profiles", "HT29", "Suppl Fig 4h-i")
    if "SliceMedianAgg_Combined" in name:
        return ("Per-slice aggregates, three acquisitions combined", "HT29",
                "input to exp4 feature selection")
    if name.startswith("grit_"):
        return ("Grit scores", name[len("grit_"):-len(".parquet")], "Suppl Fig 3c")
    if name.startswith("selected_"):
        return ("Feature-selected profiles", name[len("selected_"):-len(".parquet")],
                "Suppl Fig 3d-e")
    raise ValueError(f"no description rule for {path!r}")


records = []
for r in rows:
    kind, cond, used = describe(r["path"])
    records.append({
        "Files": f'{DEST_FOLDER}/{r["path"]}',
        "experiment": r["path"].split("/")[0],
        "experiment_description": EXPERIMENTS[r["path"].split("/")[0]],
        "data_type": kind,
        "condition": cond,
        "used_by": used,
        "download_tier": r["tier"],
        "size_bytes": r["size_bytes"],
        "sha256": r["sha256"],
    })

FileList = pd.DataFrame.from_records(records).sort_values("Files").reset_index(drop=True)
print(f"{len(FileList)} rows, {len(FileList.columns)} columns")
FileList.head()

## Check against what will actually be uploaded

Every listed file must exist in `data/` at the size the manifest records. A file list
that promises a file the upload does not contain is how a deposit ends up with dangling
entries.


In [ ]:
missing, wrong_size = [], []
for rec in records:
    rel = rec["Files"][len(DEST_FOLDER) + 1:]
    f = DATA_ROOT / rel
    if not f.is_file():
        missing.append(rel)
    elif f.stat().st_size != int(rec["size_bytes"]):
        wrong_size.append((rel, f.stat().st_size, rec["size_bytes"]))

assert not missing, f"listed but not in data/: {missing}"
assert not wrong_size, f"size differs from the manifest: {wrong_size}"

by_tier = FileList.groupby("download_tier")["size_bytes"].agg(
    files="count", bytes=lambda s: s.astype(int).sum())
by_tier["MB"] = (by_tier["bytes"] / 1e6).round(1)
print(by_tier[["files", "MB"]])
print(f"\ntotal {FileList['size_bytes'].astype(int).sum() / 1e6:.0f} MB")

## Write

CRLF line endings, matching `spher_colo52-FileList.tsv` — that file went through a
spreadsheet on its way to BIA, and keeping the two lists byte-consistent avoids a
spurious diff between them.


In [ ]:
with OUT.open("w", newline="", encoding="utf-8") as fh:
    fh.write("\t".join(FileList.columns) + "\r\n")
    for row in FileList.itertuples(index=False):
        fh.write("\t".join("" if pd.isna(v) else str(v) for v in row) + "\r\n")

print(f"wrote {OUT.resolve()}")
print(f"  {len(FileList)} files, "
      f"{FileList['size_bytes'].astype(int).sum() / 1e6:.0f} MB")
print("\nUpload the files with the layout in the Files column, i.e. "
      f"{DEST_FOLDER}/<experiment>/..., then submit the study update.")